# GF(2) and CSS Stabilizer Basics

This notebook is the Phase 1 walkthrough for the QEC/FTQC portfolio. It shows the binary linear algebra used by CSS code checks, then verifies the repetition and Steane reference codes.

## Environment check

Run this cell first. It makes the notebook work even if Jupyter starts from the `examples/` directory or VS Code selects the parent virtual environment.

In [1]:
from __future__ import annotations

import importlib.util
import subprocess
import sys
from pathlib import Path


def find_project_root(start: Path) -> Path:
    for path in (start, *start.parents):
        if (path / "pyproject.toml").exists() and (path / "qec_toolkit").is_dir():
            return path
    raise RuntimeError("Could not find qec-ftqc-portfolio project root.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

if importlib.util.find_spec("numpy") is None:
    print(f"Installing numpy into active kernel: {sys.executable}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "numpy>=1.26"])

print("Python:", sys.executable)
print("Project root:", PROJECT_ROOT)

Python: c:\Users\param\OneDrive\Desktop\QuantumComputing-Research\QEC-Research\qec-ftqc-portfolio\.venv\Scripts\python.exe
Project root: C:\Users\param\OneDrive\Desktop\QuantumComputing-Research\QEC-Research\qec-ftqc-portfolio


## Imports

In [2]:
import numpy as np

from qec_toolkit.codes.repetition import make_repetition_code
from qec_toolkit.codes.steane import make_steane_code
from qec_toolkit.verification.gf2 import (
    gf2_in_rowspace,
    gf2_nullspace,
    gf2_rank,
    gf2_rref,
    symplectic_inner,
)

print("numpy:", np.__version__)

numpy: 2.5.0


## Rank and RREF over GF(2)

Over GF(2), row addition is XOR. A matrix can have a different rank over GF(2) than over the real numbers.

In [3]:
a = np.array(
    [
        [1, 1, 0],
        [0, 1, 1],
        [1, 0, 1],
    ],
    dtype=np.uint8,
)

rref, pivots = gf2_rref(a)
print("A =")
print(a)
print("RREF_GF2(A) =")
print(rref)
print("pivot columns:", pivots)
print("rank_GF2(A):", gf2_rank(a))

A =
[[1 1 0]
 [0 1 1]
 [1 0 1]]
RREF_GF2(A) =
[[1 0 1]
 [0 1 1]
 [0 0 0]]
pivot columns: [0, 1]
rank_GF2(A): 2


## Nullspace and rowspace

The nullspace contains vectors `v` with `A @ v == 0 mod 2`. Rowspace membership tells whether a candidate vector is generated by the stabilizer rows.

In [4]:
h = np.array(
    [
        [1, 0, 1, 0],
        [0, 1, 1, 0],
    ],
    dtype=np.uint8,
)

null_basis = gf2_nullspace(h)
print("nullspace basis rows:")
print(null_basis)
print("H @ basis.T mod 2:")
print((h @ null_basis.T) % 2)

candidate = np.array([1, 1, 0, 0], dtype=np.uint8)
print("candidate in rowspace:", gf2_in_rowspace(h, candidate))

nullspace basis rows:
[[1 1 1 0]
 [0 0 0 1]]
H @ basis.T mod 2:
[[0 0]
 [0 0]]
candidate in rowspace: True


## Symplectic commutation

A Pauli vector is represented as `(x | z)`. Two Paulis commute iff their symplectic inner product is 0.

In [5]:
x0 = np.array([1, 0, 0, 0], dtype=np.uint8)
z0 = np.array([0, 0, 1, 0], dtype=np.uint8)
z1 = np.array([0, 0, 0, 1], dtype=np.uint8)

print("X0 vs Z0:", symplectic_inner(x0, z0))
print("X0 vs Z1:", symplectic_inner(x0, z1))

X0 vs Z0: 1
X0 vs Z1: 0


## Reference CSS codes

For a CSS code, validity means `H_X @ H_Z.T == 0 mod 2`, and `k = n - rank(H_X) - rank(H_Z)`.

In [6]:
for code in [make_repetition_code(3), make_steane_code()]:
    code.validate()
    print(code.summary())

3-qubit repetition: [[3, 1, 3]] | r_x=0, r_z=2 | avg_wX=0.0, avg_wZ=2.0 | CSS=OK
[[7,1,3]] Steane: [[7, 1, 3]] | r_x=3, r_z=3 | avg_wX=4.0, avg_wZ=4.0 | CSS=OK
